<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

In [53]:
solar_stat = "Max"

In [85]:
pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv").columns

Index(['apparent_zenith', 'zenith', 'apparent_elevation', 'elevation',
       'azimuth', 'equation_of_time', 'airmass_relative', 'airmass_absolute',
       'ghi', 'dni', 'dhi'],
      dtype='object')

In [3]:
to_drop = ["Latitude", "Longitude", "datetime"]
target = "UHI Index"

df = pd.concat([
    pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Final.csv"),
    pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv"),
    pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv"),
], axis=1).drop(to_drop, axis=1, errors="ignore")
feature_shape = df.shape[1]
feature_shape

191

In [165]:
from sklearn.feature_selection import SelectPercentile, f_regression, mutual_info_regression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    VotingRegressor,
    StackingRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    AdaBoostRegressor
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample

def calculate_vif_(X, thresh=5.0):
    print("Detecting multicolinearity...")
    X = X.assign(const=1)  # faster than add_constant from statsmodels
    variables = list(range(X.shape[1]))
    dropped = True
    while dropped:
        dropped = False
        vif = [variance_inflation_factor(X.iloc[:, variables].values, ix)
               for ix in range(X.iloc[:, variables].shape[1])]
        vif = vif[:-1]  # don't let the constant be removed in the loop.
        maxloc = vif.index(max(vif))
        if max(vif) > thresh:
            print('dropping \'' + X.iloc[:, variables].columns[maxloc] +
                  '\' at index: ' + str(maxloc))
            del variables[maxloc]
            dropped = True

    print('Remaining variables:')
    print(X.columns[variables[:-1]])
    return X.iloc[:, variables[:-1]]

def add_features(df):
    # Existing features
    stats = ["median", "mean", "min", "max", "var", "std"]
    epsilon = 1e-7

    count_cols = df.columns[df.columns.str.contains("count")]
    for col in count_cols:
        divider = int(col.split("_")[0].replace("m", "")[:-1])
        df[f"{col}_density_per_{divider}m"] = df[col] / divider

    for stat in stats:
        df[f"{stat}_evi_x_lwir"] = df[f"evi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_x_lwir"] = df[f"ndbi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_/_bldg_dnsty"] = df[f"ndbi_{stat}"] / df[f"building_density"].add(epsilon)
        df[f"{stat}_ndbi_/_ndwi"] = df[f"ndbi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_ndvi"] = df[f"ndbi_{stat}"] / df[f"ndvi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_evi"] = df[f"ndbi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_wvp_/_lwir"] = df[f"wvp_{stat}"] / df[f"lwir_{stat}"].add(epsilon)
        df[f"{stat}_infra_red_combo"] = df[f"nir08_{stat}"] * df[f"swir16_{stat}"] * df[f"swir22_{stat}"]
        df[f"{stat}_relative_ndvi"] = df[f"ndvi_{stat}"] / (df[f"ndvi_{stat}"].max() + epsilon)
        df[f"{stat}_relative_ndwi"] = df[f"ndwi_{stat}"] / (df[f"ndwi_{stat}"].max() + epsilon)
        df[f"{stat}_ndvi_ndwi_ratio"] = df[f"ndvi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_evi_ratio"] = df[f"ndvi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndwi_evi_ratio"] = df[f"ndwi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_ndwi_diff"] = df[f"ndvi_{stat}"] - df[f"ndwi_{stat}"]
        df[f"{stat}_ndvi_evi_diff"] = df[f"ndvi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_ndwi_evi_diff"] = df[f"ndwi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_combined_spectral_index"] = (df[f"ndvi_{stat}"] + df[f"ndwi_{stat}"] + df[f"evi_{stat}"]) / 3

        df[f"{stat}_bsi"] = (df[f'swir16_{stat}'] + df[f'swir22_{stat}'] - 2 * df[f'nir08_{stat}']) / (df[f'swir16_{stat}'] + df[f'swir22_{stat}'] + 2 * df[f'nir08_{stat}'])
        df[f"{stat}_savi"] = ((df[f'nir08_{stat}'] - df[f'red_{stat}']) * (1 + 0.5)) / (df[f'nir08_{stat}'] + df[f'red_{stat}'] + 0.5)
        df[f"{stat}_sr"] = df[f'nir08_{stat}'] / df[f'red_{stat}']
        df[f"{stat}_dsi"] =  (df[f'swir22_{stat}'] - df[f'nir08_{stat}']) / (df[f'swir22_{stat}'] + df[f'nir08_{stat}'])
        df[f"{stat}_wvi"] = df[f'wvp_{stat}'] / (df[f'swir22_{stat}'] + df[f'swir16_{stat}'] + df[f'nir08_{stat}'])

    df["mean_vci"] = (df[f'ndvi_mean'] - df.ndvi_min) / (df.ndvi_max - df.ndvi_min)
    df["median_vci"] = (df[f'ndvi_median'] - df.ndvi_min) / (df.ndvi_max - df.ndvi_min)


    # # Solar derived
    # total_solar_radiation = df["ghi"] + df["dhi"] + df["dni"]
    # df["total_solar_radiation"] = total_solar_radiation

    # df[f"total_solar_radiation_100m"] = np.pi * 100**2 * total_solar_radiation

    # df["corrected_dhi_relative"] = df["dhi"] / df["airmass_relative"]
    # df["corrected_dni_relative"] = df["dni"] / df["airmass_relative"]
    # df["corrected_ghi_relative"] = df["ghi"] / df["airmass_relative"]

    # df["corrected_dhi_absolute"] = df["dhi"] / df["airmass_absolute"]
    # df["corrected_dni_absolute"] = df["dni"] / df["airmass_absolute"]
    # df["corrected_ghi_absolute"] = df["ghi"] / df["airmass_absolute"]



    df["normalized_distance_range"] = df.distance_range / df.average_distance.add(epsilon)
    df["normalized_distance_variation"] = df.distance_variation / df.average_distance.add(epsilon)
    df["distance_building_size_interaction"] = df.nearest_building_distance * df.nearest_building_size
    df["distance_building_density_interaction"] = df.nearest_building_distance * df.building_density
    df["average_distance_squared"] = df.average_distance ** 2
    df["nearest_building_distance_squared"] = df.nearest_building_distance ** 2
    df["log_building_area_density"] = np.log(df.building_area_density.add(epsilon))
    df["log_nearest_building_distance"] = np.log(df.nearest_building_distance.add(epsilon))
    df["distance_std_to_mean_ratio"] = df.average_distance / df.average_distance.add(epsilon)
    df["distance_variation_to_range_ratio"] = df.distance_variation / df.distance_range.add(epsilon)

    # Dangerous BOCOR feature!
    return df

def create_train(df_features, target="UHI Index", train_size=0.8):
    # print("Removing duplicates...")
    # rows_before = df_features.shape[0]
    # check_dupl = df_features.columns[1:]
    # df_features = df_features.drop_duplicates(subset=check_dupl, keep='first')
    # rows_after = df_features.shape[0]
    # print(f"Removed {rows_before-rows_after} duplicate rows!")

    # ########################################
    # global index_features, building_features
    # experiment = np.concatenate((index_features, building_features))
    # try:
    #     X = df_features.drop(target, axis=1)[experiment]
    # except:
    #     X = df_features.drop(target, axis=1)
    # ########################################
    X = df_features.drop(target, axis=1)
    y = df_features[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=train_size)

    return X_train, X_test, y_train, y_test

def round1(models, train_size):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"

    # Round 1 to get pareto + 1 features
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    print(spaces, "Starting round 1", spaces)
    print(separator)
    global solar_stat
    df = pd.concat([
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Final.csv"),
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv")
    ], axis=1).drop(to_drop, axis=1, errors="ignore")

    building_features = df.iloc[:, -23:].columns
    df = df.pipe(add_features)

    X_train, X_test, y_train, y_test = create_train(df, train_size=train_size)

    # # # # # # # # # # #
    select = SelectPercentile(f_regression, percentile=30)
    select.fit(X_train, y_train)
    X_train = X_train[select.get_feature_names_out()]
    X_test = X_test[X_train.columns]
    # # # # # # # # # # #

    for model in tqdm(models):
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results)
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    if isinstance(best_model.model, (VotingRegressor, StackingRegressor)):
        print("Best model is either voting or stacking!")
        return
    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = (importance.Importance.cumsum() / importance.Importance.sum()).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return importance

def round2(models, pareto_threshold, importance, train_size):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    # Round 2 to get pareto + 1
    if importance is not None:
        pareto = importance[importance.cumulative_importance <= pareto_threshold]
        print(equals, "Pareto Features + 1", equals)
        print(pareto)

    print(separator)

    print(spaces, "Starting round 2", spaces)
    print(separator)
    global solar_stat
    df = pd.concat([
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Final.csv"),
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv")
    ], axis=1)
    df = df.pipe(add_features)

    if importance is not None:
        use_cols = [target, *pareto.Features]
        df = df.loc[:, use_cols]
    else:
        df = df.drop(to_drop, axis=1, errors="ignore")

    X_train, X_test, y_train, y_test = create_train(df, train_size=train_size)
    n_features = int(X_train.shape[1] // 3)

    ######################################################
    # X_train = pd.concat([X_train, X_test])
    # y_train = pd.concat([y_train, y_test])
    ######################################################

    for model in tqdm(models):
        if not isinstance(model["model"], (VotingRegressor, StackingRegressor)):
            model["model"].set_params(max_features=n_features)
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results.head(1))
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    if isinstance(best_model.model, (VotingRegressor, StackingRegressor)):
        print("Best model is either voting or stacking!")
        return best_model, X_train

    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = (importance.Importance.cumsum() / importance.Importance.sum()).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return best_model, X_train

# Modelling

In [166]:
max_features = int(feature_shape // 3)
basic_params = {
    "n_jobs": -1,
    "random_state": 0,
}

vote = VotingRegressor(
    [
        ("1", ExtraTreesRegressor(max_features=0.25, n_estimators=200, **basic_params)),
        ("2", ExtraTreesRegressor(max_features=0.5, n_estimators=200, **basic_params)),
        ("3", ExtraTreesRegressor(max_features=0.75, n_estimators=200, **basic_params)),
        ("4", ExtraTreesRegressor(max_features=1.0, n_estimators=200, **basic_params)),
        ("5", ExtraTreesRegressor(max_features=max_features, n_estimators=300, **basic_params)),
    ],
    n_jobs=-1
)

stack = StackingRegressor(
    [
        ("1", ExtraTreesRegressor(max_features=0.25, n_estimators=200, **basic_params)),
        ("2", ExtraTreesRegressor(max_features=0.5, n_estimators=200, **basic_params)),
        ("3", ExtraTreesRegressor(max_features=0.75, n_estimators=200, **basic_params)),
        ("4", ExtraTreesRegressor(max_features=1.0, n_estimators=200, **basic_params)),
    ],
    ExtraTreesRegressor(max_features=max_features, n_estimators=300, **basic_params),
    cv=2,
    n_jobs=-1
)

models = [
    # {"model": RandomForestRegressor(**basic_params, max_features=max_features)},
    # {"model": RandomForestRegressor(250, max_features=max_features, **basic_params)},
    # {"model": RandomForestRegressor(150, max_features=max_features, **basic_params)},
    {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=300, **basic_params)},
    {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=250, **basic_params)},
    {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=200, **basic_params)},
    # {"model": GradientBoostingRegressor(max_features=max_features, n_estimators=250, random_state=0)},
    # {"model": BaggingRegressor(max_features=max_features, n_estimators=250, **basic_params)},
    # {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=100, **basic_params)},
    # {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=300, **basic_params)},
    # {"model": vote},
    # {"model": stack},
]

importance = round1(models, train_size=0.85)

                         Starting round 1                         
------------------------------------------------------------------


100%|██████████| 3/3 [01:35<00:00, 31.74s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features=63, random_st...       1.0   0.971037
1  (ExtraTreeRegressor(max_features=63, random_st...       1.0   0.970959
2  (ExtraTreeRegressor(max_features=63, random_st...       1.0   0.970934
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features=63, n_estimators=300, n_jobs=-1,
                    random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
                     Features  Importance  cumulative_importance
0             median_distance    0.058506                   0.06
1                max_distance    0.054277                   0.11
2    average_distance_squared   

In [169]:
best_model, X_train = round2(models, pareto_threshold=0.51, importance=importance,
                             train_size=0.98)
# pareto=0.53, split=0.98 | 0.984528 # solar max Feature F_reg 25% # SCORE 0.9811
# pareto=0.53, split=0.98 | 0.984196 # solar max Feature F_reg 30% # SCORE 0.9817

================================ Pareto Features + 1 ================================
                    Features  Importance  cumulative_importance
0            median_distance    0.058506                   0.06
1               max_distance    0.054277                   0.11
2   average_distance_squared    0.051017                   0.16
3         distance_variation    0.049450                   0.21
4           average_distance    0.047762                   0.26
5             distance_range    0.045809                   0.31
6                  atran_min    0.039025                   0.35
7           airmass_absolute    0.029243                   0.38
8            apparent_zenith    0.027364                   0.40
9         apparent_elevation    0.027192                   0.43
10                    zenith    0.026510                   0.46
11         coast_aerosol_max    0.024871                   0.48
12          airmass_relative    0.024266                   0.51
------------------

100%|██████████| 3/3 [00:13<00:00,  4.62s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features=4, random_sta...       1.0   0.983925
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features=4, n_estimators=250, n_jobs=-1, random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
                    Features  Importance  cumulative_importance
0               max_distance    0.085722                   0.09
1         distance_variation    0.083494                   0.17
2             distance_range    0.082535                   0.25
3   average_distance_squared    0.081449                   0.33
4           airmass_relative    0.079831                   0.41
5           

# Predicting Submission

In [170]:
def create_submission(filename: str, model):
    global solar_stat
    sub_df = pd.concat([
        pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Final.csv"),
        pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Building_Features.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/Submission_SolarData{solar_stat}SinceTrainTime.csv"),
    ], axis=1)

    final_df = sub_df[["Latitude", "Longitude"]].copy()
    print("Predicting", sub_df.shape[0], "rows...")

    ############################
    sub_df = add_features(sub_df)

    # # # # # Comment if not used!
    # sub_df.scl_median = sub_df.scl_median.map(scl_mapping)
    # scl_ohe = ohe.transform(sub_df.loc[:, ["scl_median"]])
    # scl_ohe = pd.DataFrame(scl_ohe, columns=ohe.get_feature_names_out(["scl_median"]))
    # sub_df = pd.concat([sub_df.drop("scl_median", axis=1), scl_ohe], axis=1)

    to_predict = sub_df.loc[:, X_train.columns]

    print("Predicting...")
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return
create_submission(
    f"DynamicMaxFeatures_MoreNewFeats{solar_stat}_FeatureMI20pctl_v9.csv",
    best_model.model
)

Predicting 1040 rows...
Predicting...
Done!


---

In [162]:
!rm /content/Dynamic*.csv